# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


# Model Building

In [1]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

In [2]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

## Data Registration

In [3]:
os.makedirs("tourism_project/data", exist_ok=True)

Upload `tourism.csv` into `tourism_project/data/` (or keep it in the current working directory and the code below will copy it).

Before running the Hugging Face upload steps, set these environment variables:
- `HF_TOKEN`: Hugging Face write token
- `HF_DATASET_REPO`: dataset repo id like `username/visit-with-us-dataset`
- `HF_MODEL_REPO`: model repo id like `username/visit-with-us-model`
- `HF_SPACE_REPO`: space repo id like `username/visit-with-us-space`

## Data Preparation

In [4]:
# Install dependencies (run once)
# %pip install -q pandas numpy scikit-learn joblib mlflow huggingface_hub

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, hf_hub_download

BASE_DIR = Path("tourism_project")
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Config from environment variables (do not hardcode secrets)
HF_TOKEN = os.getenv("HF_TOKEN")
HF_DATASET_REPO = os.getenv("HF_DATASET_REPO", "")

# Locate dataset and copy into the project data folder if needed
source_csv = Path("tourism.csv")
if not source_csv.exists():
    raise FileNotFoundError("tourism.csv not found in current directory. Place it here and rerun.")

raw_csv_path = DATA_DIR / "tourism.csv"
if not raw_csv_path.exists():
    raw_csv_path.write_bytes(source_csv.read_bytes())

print(f"Raw dataset path: {raw_csv_path.resolve()}")

# 1) Register raw data to HF dataset repo
api = HfApi(token=HF_TOKEN) if HF_TOKEN else None

if api and HF_DATASET_REPO:
    api.create_repo(repo_id=HF_DATASET_REPO, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(raw_csv_path),
        path_in_repo="raw/tourism.csv",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
    )
    print(f"Uploaded raw dataset to: https://huggingface.co/datasets/{HF_DATASET_REPO}")
else:
    print("Skipping HF upload for raw data. Set HF_TOKEN and HF_DATASET_REPO to enable.")

# 2) Load dataset directly from HF (if available), else local copy
if api and HF_DATASET_REPO:
    downloaded_raw = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename="raw/tourism.csv",
        repo_type="dataset",
        token=HF_TOKEN,
    )
    df = pd.read_csv(downloaded_raw)
    print("Loaded dataset from Hugging Face dataset hub.")
else:
    df = pd.read_csv(raw_csv_path)
    print("Loaded dataset from local project data folder.")

print(df.shape)
print(df.head())

# 3) Data cleaning and preprocessing-ready dataset
clean_df = df.copy()
clean_df.columns = [c.strip() for c in clean_df.columns]

# Remove unnecessary ID column
if "CustomerID" in clean_df.columns:
    clean_df = clean_df.drop(columns=["CustomerID"])

# Remove duplicate records
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

# Handle missing values with sensible defaults
num_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = clean_df.select_dtypes(exclude=[np.number]).columns.tolist()

for c in num_cols:
    clean_df[c] = clean_df[c].fillna(clean_df[c].median())

for c in cat_cols:
    clean_df[c] = clean_df[c].fillna(clean_df[c].mode().iloc[0] if not clean_df[c].mode().empty else "Unknown")

# 4) Train-test split
if "ProdTaken" not in clean_df.columns:
    raise ValueError("Target column 'ProdTaken' is missing from dataset.")

train_df, test_df = train_test_split(
    clean_df,
    test_size=0.2,
    random_state=42,
    stratify=clean_df["ProdTaken"],
)

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

# 5) Upload processed splits back to HF dataset repo
if api and HF_DATASET_REPO:
    api.upload_file(
        path_or_fileobj=str(train_path),
        path_in_repo="processed/train.csv",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
    )
    api.upload_file(
        path_or_fileobj=str(test_path),
        path_in_repo="processed/test.csv",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
    )
    print("Uploaded processed train/test data to Hugging Face dataset hub.")
else:
    print("Skipping HF upload for processed data. Set HF_TOKEN and HF_DATASET_REPO to enable.")

# 6) Save a reusable script for GitHub Actions pipeline
script_path = BASE_DIR / "data_preparation.py"
script_path.write_text(
    """import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, hf_hub_download

BASE_DIR = Path(\"tourism_project\")
DATA_DIR = BASE_DIR / \"data\"
DATA_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.getenv(\"HF_TOKEN\")
HF_DATASET_REPO = os.getenv(\"HF_DATASET_REPO\", \"\")

raw_csv_path = DATA_DIR / \"tourism.csv\"
if not raw_csv_path.exists() and Path(\"tourism.csv\").exists():
    raw_csv_path.write_bytes(Path(\"tourism.csv\").read_bytes())

if not raw_csv_path.exists():
    raise FileNotFoundError(\"tourism.csv is required\")

api = HfApi(token=HF_TOKEN) if HF_TOKEN else None
if api and HF_DATASET_REPO:
    api.create_repo(repo_id=HF_DATASET_REPO, repo_type=\"dataset\", exist_ok=True)
    api.upload_file(path_or_fileobj=str(raw_csv_path), path_in_repo=\"raw/tourism.csv\", repo_id=HF_DATASET_REPO, repo_type=\"dataset\")
    downloaded_raw = hf_hub_download(repo_id=HF_DATASET_REPO, filename=\"raw/tourism.csv\", repo_type=\"dataset\", token=HF_TOKEN)
    df = pd.read_csv(downloaded_raw)
else:
    df = pd.read_csv(raw_csv_path)

clean_df = df.copy()
clean_df.columns = [c.strip() for c in clean_df.columns]
if \"CustomerID\" in clean_df.columns:
    clean_df = clean_df.drop(columns=[\"CustomerID\"])
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

num_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = clean_df.select_dtypes(exclude=[np.number]).columns.tolist()
for c in num_cols:
    clean_df[c] = clean_df[c].fillna(clean_df[c].median())
for c in cat_cols:
    mode_vals = clean_df[c].mode()
    clean_df[c] = clean_df[c].fillna(mode_vals.iloc[0] if not mode_vals.empty else \"Unknown\")

train_df, test_df = train_test_split(clean_df, test_size=0.2, random_state=42, stratify=clean_df[\"ProdTaken\"])
train_path = DATA_DIR / \"train.csv\"
test_path = DATA_DIR / \"test.csv\"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

if api and HF_DATASET_REPO:
    api.upload_file(path_or_fileobj=str(train_path), path_in_repo=\"processed/train.csv\", repo_id=HF_DATASET_REPO, repo_type=\"dataset\")
    api.upload_file(path_or_fileobj=str(test_path), path_in_repo=\"processed/test.csv\", repo_id=HF_DATASET_REPO, repo_type=\"dataset\")

print(\"Data preparation complete.\")
""",
    encoding="utf-8",
)
print(f"Saved script: {script_path}")

Raw dataset path: /Users/jaiwala/Great learning/tourism-package-prediction/tourism_project/data/tourism.csv
Skipping HF upload for raw data. Set HF_TOKEN and HF_DATASET_REPO to enable.
Loaded dataset from local project data folder.
(4128, 21)
   Unnamed: 0  CustomerID  ProdTaken   Age    TypeofContact  CityTier  \
0           0      200000          1  41.0     Self Enquiry         3   
1           1      200001          0  49.0  Company Invited         1   
2           2      200002          1  37.0     Self Enquiry         1   
3           3      200003          0  33.0  Company Invited         1   
4           5      200005          0  32.0  Company Invited         1   

   DurationOfPitch   Occupation  Gender  NumberOfPersonVisiting  ...  \
0              6.0     Salaried  Female                       3  ...   
1             14.0     Salaried    Male                       3  ...   
2              8.0  Free Lancer    Male                       3  ...   
3              9.0     Salarie

## Model Training and Registration with Experimentation Tracking

In [5]:
import os
import json
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

BASE_DIR = Path("tourism_project")
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "model_building"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.getenv("HF_TOKEN")
HF_DATASET_REPO = os.getenv("HF_DATASET_REPO", "")
HF_MODEL_REPO = os.getenv("HF_MODEL_REPO", "")

api = HfApi(token=HF_TOKEN) if HF_TOKEN else None

# Load train/test from HF hub if available, else local
if api and HF_DATASET_REPO:
    train_file = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename="processed/train.csv",
        repo_type="dataset",
        token=HF_TOKEN,
    )
    test_file = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename="processed/test.csv",
        repo_type="dataset",
        token=HF_TOKEN,
    )
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    print("Loaded processed train/test from Hugging Face dataset hub.")
else:
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")
    print("Loaded processed train/test from local folder.")

X_train = train_df.drop(columns=["ProdTaken"])
y_train = train_df["ProdTaken"]
X_test = test_df.drop(columns=["ProdTaken"])
y_test = test_df["ProdTaken"]

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]), num_cols),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=42)),
    ]
)

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_split": [2, 5],
}

mlflow.set_tracking_uri(f"file:{(MODEL_DIR / 'mlruns').resolve()}")
mlflow.set_experiment("tourism-wellness-package")

with mlflow.start_run(run_name="rf-gridsearch"):
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred)),
        "recall": float(recall_score(y_test, y_pred)),
        "f1": float(f1_score(y_test, y_pred)),
        "roc_auc": float(roc_auc_score(y_test, y_proba)),
    }

    # Log all tuned parameters and best model metrics
    for k, v in grid.best_params_.items():
        mlflow.log_param(k, v)
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    cv_results = pd.DataFrame(grid.cv_results_)
    cv_results_path = MODEL_DIR / "cv_results.csv"
    cv_results.to_csv(cv_results_path, index=False)
    mlflow.log_artifact(str(cv_results_path))

    model_path = MODEL_DIR / "best_model.joblib"
    metrics_path = MODEL_DIR / "metrics.json"
    joblib.dump(best_model, model_path)
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("Best Parameters:", grid.best_params_)
print("Test Metrics:", metrics)

# Register best model in Hugging Face model hub
if api and HF_MODEL_REPO:
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(model_path),
        path_in_repo="best_model.joblib",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
    )
    api.upload_file(
        path_or_fileobj=str(metrics_path),
        path_in_repo="metrics.json",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
    )
    api.upload_file(
        path_or_fileobj=str(cv_results_path),
        path_in_repo="cv_results.csv",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
    )
    print(f"Registered model artifacts at: https://huggingface.co/{HF_MODEL_REPO}")
else:
    print("Skipping HF model registration. Set HF_TOKEN and HF_MODEL_REPO to enable.")

# Save reusable training script for GitHub Actions
training_script = BASE_DIR / "train_and_register.py"
training_script.write_text(
    """import json
import os
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

BASE_DIR = Path(\"tourism_project\")
DATA_DIR = BASE_DIR / \"data\"
MODEL_DIR = BASE_DIR / \"model_building\"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.getenv(\"HF_TOKEN\")
HF_DATASET_REPO = os.getenv(\"HF_DATASET_REPO\", \"\")
HF_MODEL_REPO = os.getenv(\"HF_MODEL_REPO\", \"\")
api = HfApi(token=HF_TOKEN) if HF_TOKEN else None

if api and HF_DATASET_REPO:
    train_file = hf_hub_download(repo_id=HF_DATASET_REPO, filename=\"processed/train.csv\", repo_type=\"dataset\", token=HF_TOKEN)
    test_file = hf_hub_download(repo_id=HF_DATASET_REPO, filename=\"processed/test.csv\", repo_type=\"dataset\", token=HF_TOKEN)
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
else:
    train_df = pd.read_csv(DATA_DIR / \"train.csv\")
    test_df = pd.read_csv(DATA_DIR / \"test.csv\")

X_train = train_df.drop(columns=[\"ProdTaken\"])
y_train = train_df[\"ProdTaken\"]
X_test = test_df.drop(columns=[\"ProdTaken\"])
y_test = test_df[\"ProdTaken\"]

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    (\"num\", Pipeline([(\"imputer\", SimpleImputer(strategy=\"median\"))]), num_cols),
    (\"cat\", Pipeline([(\"imputer\", SimpleImputer(strategy=\"most_frequent\")), (\"encoder\", OneHotEncoder(handle_unknown=\"ignore\"))]), cat_cols),
])

pipeline = Pipeline([
    (\"preprocessor\", preprocessor),
    (\"model\", RandomForestClassifier(random_state=42)),
])

param_grid = {
    \"model__n_estimators\": [100, 200],
    \"model__max_depth\": [None, 8, 12],
    \"model__min_samples_split\": [2, 5],
}

mlflow.set_tracking_uri(f\"file:{(MODEL_DIR / 'mlruns').resolve()}\")
mlflow.set_experiment(\"tourism-wellness-package\")

with mlflow.start_run(run_name=\"rf-gridsearch\"):
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring=\"f1\", n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    metrics = {
        \"accuracy\": float(accuracy_score(y_test, y_pred)),
        \"precision\": float(precision_score(y_test, y_pred)),
        \"recall\": float(recall_score(y_test, y_pred)),
        \"f1\": float(f1_score(y_test, y_pred)),
        \"roc_auc\": float(roc_auc_score(y_test, y_proba)),
    }

    for k, v in grid.best_params_.items():
        mlflow.log_param(k, v)
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    cv_results = pd.DataFrame(grid.cv_results_)
    cv_results_path = MODEL_DIR / \"cv_results.csv\"
    cv_results.to_csv(cv_results_path, index=False)
    mlflow.log_artifact(str(cv_results_path))

    model_path = MODEL_DIR / \"best_model.joblib\"
    metrics_path = MODEL_DIR / \"metrics.json\"
    joblib.dump(best_model, model_path)
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding=\"utf-8\")

if api and HF_MODEL_REPO:
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type=\"model\", exist_ok=True)
    api.upload_file(path_or_fileobj=str(model_path), path_in_repo=\"best_model.joblib\", repo_id=HF_MODEL_REPO, repo_type=\"model\")
    api.upload_file(path_or_fileobj=str(metrics_path), path_in_repo=\"metrics.json\", repo_id=HF_MODEL_REPO, repo_type=\"model\")
    api.upload_file(path_or_fileobj=str(cv_results_path), path_in_repo=\"cv_results.csv\", repo_id=HF_MODEL_REPO, repo_type=\"model\")

print(\"Training and registration complete.\")
""",
    encoding="utf-8",
)
print(f"Saved script: {training_script}")

Loaded processed train/test from local folder.
Fitting 5 folds for each of 12 candidates, totalling 60 fits


Best Parameters: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Test Metrics: {'accuracy': 0.9019370460048426, 'precision': 0.9148936170212766, 'recall': 0.5408805031446541, 'f1': 0.6798418972332015, 'roc_auc': 0.9566254608544784}
Skipping HF model registration. Set HF_TOKEN and HF_MODEL_REPO to enable.
Saved script: tourism_project/train_and_register.py


# Deployment

## Dockerfile

In [6]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [7]:
%%writefile tourism_project/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Overwriting tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [8]:
%%writefile tourism_project/deployment/app.py
import os
import pandas as pd
import joblib
import streamlit as st
from huggingface_hub import hf_hub_download

st.set_page_config(page_title="Wellness Package Predictor", page_icon="🌍")
st.title("Visit with Us - Wellness Package Purchase Prediction")

HF_MODEL_REPO = os.getenv("HF_MODEL_REPO", "")
HF_TOKEN = os.getenv("HF_TOKEN")

@st.cache_resource
def load_model():
    if HF_MODEL_REPO:
        model_file = hf_hub_download(
            repo_id=HF_MODEL_REPO,
            filename="best_model.joblib",
            repo_type="model",
            token=HF_TOKEN,
        )
    else:
        model_file = "tourism_project/model_building/best_model.joblib"
    return joblib.load(model_file)

model = load_model()

st.subheader("Enter customer details")

input_data = {
    "Age": st.number_input("Age", min_value=18, max_value=90, value=35),
    "TypeofContact": st.selectbox("Type of Contact", ["Self Enquiry", "Company Invited"]),
    "CityTier": st.selectbox("City Tier", [1, 2, 3]),
    "Occupation": st.selectbox("Occupation", ["Salaried", "Small Business", "Free Lancer", "Large Business"]),
    "Gender": st.selectbox("Gender", ["Male", "Female"]),
    "NumberOfPersonVisiting": st.number_input("Number of Persons Visiting", min_value=1, max_value=10, value=2),
    "PreferredPropertyStar": st.selectbox("Preferred Property Star", [1, 2, 3, 4, 5]),
    "MaritalStatus": st.selectbox("Marital Status", ["Single", "Married", "Divorced"]),
    "NumberOfTrips": st.number_input("Number of Trips per Year", min_value=0, max_value=50, value=3),
    "Passport": st.selectbox("Passport", [0, 1]),
    "OwnCar": st.selectbox("Own Car", [0, 1]),
    "NumberOfChildrenVisiting": st.number_input("Children Visiting (<5 years)", min_value=0, max_value=6, value=0),
    "Designation": st.selectbox("Designation", ["Manager", "Senior Manager", "AVP", "VP", "Executive"]),
    "MonthlyIncome": st.number_input("Monthly Income", min_value=1000, max_value=500000, value=50000),
    "PitchSatisfactionScore": st.slider("Pitch Satisfaction Score", min_value=1, max_value=5, value=3),
    "ProductPitched": st.selectbox("Product Pitched", ["Basic", "Standard", "Deluxe", "Super Deluxe", "King"]),
    "NumberOfFollowups": st.number_input("Number of Follow-ups", min_value=0, max_value=10, value=2),
    "DurationOfPitch": st.number_input("Duration of Pitch (minutes)", min_value=1, max_value=200, value=30),
}

if st.button("Predict"):
    input_df = pd.DataFrame([input_data])  # required rubric step: save inputs into a dataframe
    st.write("Input DataFrame", input_df)

    pred = model.predict(input_df)[0]
    prob = model.predict_proba(input_df)[0][1]

    st.success(f"Prediction (ProdTaken): {int(pred)}")
    st.info(f"Probability of purchase: {prob:.2%}")

Overwriting tourism_project/deployment/app.py


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [9]:
%%writefile tourism_project/deployment/requirements.txt
pandas
numpy
scikit-learn
joblib
streamlit
huggingface_hub
mlflow

Overwriting tourism_project/deployment/requirements.txt


# Hosting

In [10]:
%%writefile tourism_project/deployment/deploy_to_space.py
import os
from huggingface_hub import HfApi

HF_TOKEN = os.getenv("HF_TOKEN")
HF_SPACE_REPO = os.getenv("HF_SPACE_REPO", "")

if not HF_TOKEN:
    raise EnvironmentError("HF_TOKEN is required for deployment.")
if not HF_SPACE_REPO:
    raise EnvironmentError("HF_SPACE_REPO is required, e.g., username/visit-with-us-space")

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_SPACE_REPO, repo_type="space", space_sdk="streamlit", exist_ok=True)

api.upload_folder(
    folder_path="tourism_project/deployment",
    repo_id=HF_SPACE_REPO,
    repo_type="space",
)

print(f"Deployment files pushed to: https://huggingface.co/spaces/{HF_SPACE_REPO}")

Overwriting tourism_project/deployment/deploy_to_space.py


# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

%%writefile tourism_project/.github/workflows/pipeline.yml
name: Tourism MLOps Pipeline

on:
  push:
    branches: ["main"]
  workflow_dispatch:

jobs:
  data-registration-and-prep:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/github_actions_requirements.txt

      - name: Run data preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_DATASET_REPO: ${{ secrets.HF_DATASET_REPO }}
        run: python tourism_project/data_preparation.py

  model-training-and-registration:
    runs-on: ubuntu-latest
    needs: data-registration-and-prep
    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/github_actions_requirements.txt

      - name: Train and register model
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_DATASET_REPO: ${{ secrets.HF_DATASET_REPO }}
          HF_MODEL_REPO: ${{ secrets.HF_MODEL_REPO }}
        run: python tourism_project/train_and_register.py

  deploy-space:
    runs-on: ubuntu-latest
    needs: model-training-and-registration
    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/github_actions_requirements.txt

      - name: Push deployment files to Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_SPACE_REPO: ${{ secrets.HF_SPACE_REPO }}
        run: python tourism_project/deployment/deploy_to_space.py

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

In [11]:
%%writefile tourism_project/github_actions_requirements.txt
pandas
numpy
scikit-learn
joblib
mlflow
huggingface_hub
streamlit

Overwriting tourism_project/github_actions_requirements.txt


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [12]:
# Run these commands in Colab/local terminal after replacing placeholders
print("""
# Clone your GitHub repository
git clone https://github.com/<github-username>/<repo-name>.git

# Move project folder into repo
mv tourism_project <repo-name>/

# Go to repo and push
cd <repo-name>
git add .
git commit -m \"Add tourism MLOps pipeline\"
git push origin main
""")


# Clone your GitHub repository
git clone https://github.com/<github-username>/<repo-name>.git

# Move project folder into repo
mv tourism_project <repo-name>/

# Go to repo and push
cd <repo-name>
git add .
git commit -m "Add tourism MLOps pipeline"
git push origin main



In [13]:
print("Use the repository terminal for push operations. Avoid embedding GitHub tokens in notebook cells for security reasons.")

Use the repository terminal for push operations. Avoid embedding GitHub tokens in notebook cells for security reasons.


# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

- GitHub Repository Link: `https://github.com/<your-username>/<your-repo-name>`
- Add screenshot: repository folder structure
- Add screenshot: executed GitHub Actions workflow (`pipeline.yml`)

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

- Hugging Face Space Link: `https://huggingface.co/spaces/<your-username>/<your-space-name>`
- Add screenshot: running Streamlit app home screen
- Add screenshot: sample prediction output

<font size=6 color="navyblue">Power Ahead!</font>
___